In [1]:
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

In [2]:
def add_random_missing(df, columns, ratio=0.1, random_state=42):
    df_noisy = df.copy()
    np.random.seed(random_state)
    
    for col in columns:
        n = int(len(df_noisy) * ratio)
        idx = np.random.choice(df_noisy.index, n, replace=False)
        df_noisy.loc[idx, col] = np.nan
    
    return df_noisy


def add_random_outlier(df, columns, multiplier=3, random_state=42):
    df_noisy = df.copy()
    np.random.seed(random_state)
    
    for col in columns:
        n = int(len(df_noisy) * 0.1)
        idx = np.random.choice(df_noisy.index, n, replace=False)
        df_noisy.loc[idx, col] = df_noisy.loc[idx, col] * multiplier
    
    return df_noisy

In [3]:
def evaluate_sklearn_dataset(df, target_col, dataset_name, scenario_name):
    df_eval = df.copy()
    
    # Eksik değerleri medyan ile doldur
    feature_cols = [col for col in df_eval.columns if col != target_col]
    df_eval[feature_cols] = df_eval[feature_cols].fillna(df_eval[feature_cols].median())
    
    X = df_eval.drop(columns=[target_col])
    y = df_eval[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )
    
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    return {
        "Dataset": dataset_name,
        "Scenario": scenario_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    }

In [4]:
# Breast Cancer dataset
breast = load_breast_cancer()
df_breast = pd.DataFrame(breast.data, columns=breast.feature_names)
df_breast["target"] = breast.target

# Wine dataset
wine = load_wine()
df_wine = pd.DataFrame(wine.data, columns=wine.feature_names)
df_wine["target"] = wine.target

print("Breast Cancer shape:", df_breast.shape)
print("Wine shape:", df_wine.shape)

Breast Cancer shape: (569, 31)
Wine shape: (178, 14)


In [5]:
breast_noise_cols = [
    "mean radius",
    "mean texture",
    "mean area"
]

wine_noise_cols = [
    "alcohol",
    "malic_acid",
    "color_intensity"
]

In [6]:
extra_results = []

# =========================
# Breast Cancer
# =========================

extra_results.append(
    evaluate_sklearn_dataset(df_breast, "target", "Breast Cancer", "Original")
)

df_breast_missing = add_random_missing(df_breast, breast_noise_cols, ratio=0.1)
extra_results.append(
    evaluate_sklearn_dataset(df_breast_missing, "target", "Breast Cancer", "Missing Noise")
)

df_breast_outlier = add_random_outlier(df_breast, breast_noise_cols, multiplier=3)
extra_results.append(
    evaluate_sklearn_dataset(df_breast_outlier, "target", "Breast Cancer", "Outlier Noise")
)

df_breast_both = add_random_outlier(df_breast_missing, breast_noise_cols, multiplier=3)
extra_results.append(
    evaluate_sklearn_dataset(df_breast_both, "target", "Breast Cancer", "Missing + Outlier Noise")
)


# =========================
# Wine
# =========================

extra_results.append(
    evaluate_sklearn_dataset(df_wine, "target", "Wine", "Original")
)

df_wine_missing = add_random_missing(df_wine, wine_noise_cols, ratio=0.1)
extra_results.append(
    evaluate_sklearn_dataset(df_wine_missing, "target", "Wine", "Missing Noise")
)

df_wine_outlier = add_random_outlier(df_wine, wine_noise_cols, multiplier=3)
extra_results.append(
    evaluate_sklearn_dataset(df_wine_outlier, "target", "Wine", "Outlier Noise")
)

df_wine_both = add_random_outlier(df_wine_missing, wine_noise_cols, multiplier=3)
extra_results.append(
    evaluate_sklearn_dataset(df_wine_both, "target", "Wine", "Missing + Outlier Noise")
)


extra_results_df = pd.DataFrame(extra_results)
extra_results_df

,Dataset,Scenario,Accuracy,Precision,Recall,F1 Score
0,Breast Cancer,Original,0.956140,0.956073,0.956140,0.956027
1,Breast Cancer,Missing Noise,0.947368,0.947368,0.947368,0.947368
2,Breast Cancer,Outlier Noise,0.947368,0.947368,0.947368,0.947368
3,Breast Cancer,Missing + Outlier Noise,0.947368,0.947368,0.947368,0.947368
4,Wine,Original,1.000000,1.000000,1.000000,1.000000
5,Wine,Missing Noise,1.000000,1.000000,1.000000,1.000000
6,Wine,Outlier Noise,1.000000,1.000000,1.000000,1.000000
7,Wine,Missing + Outlier Noise,1.000000,1.000000,1.000000,1.000000


In [7]:
extra_results_with_metrics = extra_results_df.copy()

extra_results_with_metrics["Original F1"] = extra_results_with_metrics.groupby("Dataset")["F1 Score"].transform("first")

extra_results_with_metrics["F1 Drop"] = (
    extra_results_with_metrics["Original F1"] - extra_results_with_metrics["F1 Score"]
)

extra_results_with_metrics["F1 Drop (%)"] = (
    extra_results_with_metrics["F1 Drop"] / extra_results_with_metrics["Original F1"] * 100
)

extra_results_with_metrics["Robustness Index"] = (
    extra_results_with_metrics["F1 Score"] / extra_results_with_metrics["Original F1"]
)

extra_results_with_metrics = extra_results_with_metrics.round(4)

extra_results_with_metrics

,Dataset,Scenario,Accuracy,Precision,Recall,F1 Score,Original F1,F1 Drop,F1 Drop (%),Robustness Index
0,Breast Cancer,Original,0.9561,0.9561,0.9561,0.9560,0.956,0.0000,0.0000,1.0000
1,Breast Cancer,Missing Noise,0.9474,0.9474,0.9474,0.9474,0.956,0.0087,0.9057,0.9909
2,Breast Cancer,Outlier Noise,0.9474,0.9474,0.9474,0.9474,0.956,0.0087,0.9057,0.9909
3,Breast Cancer,Missing + Outlier Noise,0.9474,0.9474,0.9474,0.9474,0.956,0.0087,0.9057,0.9909
4,Wine,Original,1.0000,1.0000,1.0000,1.0000,1.000,0.0000,0.0000,1.0000
5,Wine,Missing Noise,1.0000,1.0000,1.0000,1.0000,1.000,0.0000,0.0000,1.0000
6,Wine,Outlier Noise,1.0000,1.0000,1.0000,1.0000,1.000,0.0000,0.0000,1.0000
7,Wine,Missing + Outlier Noise,1.0000,1.0000,1.0000,1.0000,1.000,0.0000,0.0000,1.0000


In [8]:
extra_results_with_metrics.to_csv("extra_dataset_validation_results.csv", index=False)

In [9]:
import os

[file for file in os.listdir() if "extra" in file]

['extra_dataset_validation_results.csv']